# 5. Test LoRA

Kiem tra checkpoint da train.

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {Path.cwd()}")

In [ ]:
import torch
from diffusers import FluxPipeline
from pathlib import Path
import os

model_id = "black-forest-labs/FLUX.2-klein-base-9B"
project_name = "db9_toolkit_trainner"
checkpoint_dir = Path("outputs") / project_name
checkpoints = sorted(checkpoint_dir.rglob("*.safetensors"), key=lambda p: p.stat().st_mtime)
if not checkpoints:
    raise FileNotFoundError(f"No .safetensors checkpoints found under {checkpoint_dir}. Run training first.")
lora_path = str(checkpoints[-1])
print(f"Loading LoRA checkpoint: {lora_path}")

pipe = FluxPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)
pipe.load_lora_weights(lora_path)
pipe.to("cuda")

prompt = "A portrait of a person in DB9 style, highly detailed"
image = pipe(
    prompt,
    num_inference_steps=28,
    guidance_scale=1.0,  # Flux does not use traditional CFG
    height=1536,
    width=1536,
).images[0]

os.makedirs("outputs/test", exist_ok=True)
image.save("outputs/test/test_result.png")
print("Saved image to outputs/test/test_result.png")
image